# Demo 01 — execution model: lazy → action → stages/tasks

Цель: построить DataFrame pipeline, предсказать shuffle до запуска и связать partitions с tasks. Spark application UI доступен на [localhost:14040](http://localhost:14040).

In [ ]:
from pyspark.sql import functions as F
from mentor_spark_lab.notebook_support import create_spark_session, explain_as_text

spark = create_spark_session("lesson04-notebook-execution-model")
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}; master={spark.sparkContext.master}")
print(f"defaultParallelism={spark.sparkContext.defaultParallelism}")

## 1. Lazy transformations

Следующая ячейка только строит lineage. До `count`, `show`, `collect` или записи полноценный job не запускается.

In [ ]:
events = (
    spark.range(0, 100_000, numPartitions=8)
    .select(
        F.col("id").alias("event_id"),
        (F.col("id") % 1_000).alias("customer_id"),
        F.when(F.col("id") % 4 == 0, "purchase").otherwise("view").alias("event_type"),
        ((F.col("id") % 200).cast("double") + F.lit(0.99)).cast("decimal(10,2)").alias("amount"),
    )
)
revenue_by_bucket = (
    events.filter(F.col("event_type") == "purchase")
    .withColumn("customer_bucket", F.col("customer_id") % 5)
    .groupBy("customer_bucket")
    .agg(F.count("event_id").alias("orders"), F.sum("amount").alias("revenue"))
    .orderBy("customer_bucket")
)
print("Pipeline declared; no result has been materialized yet.")

## 2. Evidence before action

`groupBy` и глобальная сортировка — wide transformations. Поэтому до запуска ожидаем `Exchange`.

In [ ]:
plan = explain_as_text(revenue_by_bucket)
print(plan)
assert "Exchange" in plan, "Expected a shuffle boundary in the physical plan"

In [ ]:
# Action: теперь Spark создаёт job, stages и tasks.
rows = revenue_by_bucket.collect()  # collect безопасен только потому, что после агрегации здесь 5 строк.
assert len(rows) == 5
rows

## 3. Partition → task mapping

Посчитаем строки на partition. В UI сопоставь количество partition с количеством tasks в scan stage.

In [ ]:
partition_distribution = (
    events.withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id")
)
partition_rows = partition_distribution.collect()
assert len(partition_rows) == 8
partition_rows

### Самостоятельный эксперимент

Измени `numPartitions=8` на `2` и `32`. Сравни число tasks и время; не делай вывод по одному прогону — сначала объясни ожидаемую стоимость scheduling overhead.

In [ ]:
spark.stop()
print("PASS execution_model_demo")